# 5. An SN2 transition-state search
**Kernel:** AIMNet2, with Sella installed. Complete the README preflight and use a fresh kernel.

**Learning goals:** construct a transition-state guess from a constrained scan; refine a saddle;
check the unstable vibrational mode; optionally follow both reaction-path directions.

![SN2 reaction](SN2.png)

We model **Br- + CH3Cl → CH3Br + Cl-**, with total charge **-1**. This is a gas-phase model
calculation. Solvent effects and free-energy corrections are outside this exercise.
The supplied older trajectories used different settings and are not validation of this revised calculation.

**Predict:** which bond forms and which bond breaks? What motion should the unstable mode show?


In [4]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
sys.path.insert(0,str(Path.cwd().parent))
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read, write
from ase.visualize import view

from demo_tools.helpers import start_exercise

_, OUTPUT = start_exercise()


Python: /home/nchopper/anaconda3/envs/aimnet2/bin/python
Inputs: /home/nchopper/mlip-demo/workshop_demo/TransitionStates
New results: /home/nchopper/mlip-demo/workshop_demo/TransitionStates/results/20260923-162822-535507


## 1. Check the input
The six-atom XYZ uses C=0, Cl=1, Br=5. The neutral setting from the original demonstration has been
changed to -1 for the stated anionic reaction. This changes the potential-energy surface and requires
fresh calculations; never reuse the original vibration cache.


In [3]:
'''
C        0.00000000       0.00000000       0.00000000
Cl       1.78400000       0.00000000       0.00000000
H       -0.34400000       1.02900000      -0.00000000
H       -0.34400000      -0.51500000      -0.89100000
H       -0.34400000      -0.51500000       0.89100000
Br      -3.00000000       0.00000000       0.00000000
'''


molecule = read('sn2_initial_structure.xyz')
view(molecule, viewer='x3d')  # Visualize the molecule after reading it from the file


## 2. Scan the approaching C-Br distance
Fix C and Br at each separation and relax the other atoms. This is a constrained path used to find
a guess, not proof of a minimum-energy reaction path. A scan may fail to show an interior maximum;
if so, inspect the geometries and adjust the scan instead of selecting an endpoint automatically.


In [ ]:
# We need to get an initial guess for a transition state structure.
#  This can be done by manipulating the positions of the atoms in the molecule.
#  For example, we can move the nucleophile closer to the carbon atom that is being attacked.
from aimnet.calculators import AIMNet2ASE
calculator = AIMNet2ASE(
    'aimnet2-2025',  # Specify the AIMNet model to use
    charge = -1  # Specify the charge of the molecule
)

# Br - C - Cl is currently aligned along x.  We want to fix these atoms atoms along this line
#  We will also fix the position of the carbon entirely
#  Carbon index = 0
#  Chlorine index = 1
#  Bromine index = 5
from ase.constraints import FixedLine, FixAtoms, FixInternals
cbr_bond_indices = [0,5]
cbr_constraint = FixInternals(bonds=[3.0, *cbr_bond_indices])
molecule.set_constraint(cbr_constraint)


In [ ]:
# Attach calculator and move Br closer to C iteratively
molecule.calc = calculator

res = []
for cbr_distance in np.linspace(3.0, 1.7, 14):
    # Reduce the C-Br distance by 0.1 Å in each step
    molecule.set_distance(0, 5, cbr_distance, fix=0)
    molecule.set_constraint()
    constraint = FixAtoms(indices=[0, 5])
    molecule.set_constraint(constraint)
    dyn = BFGS(molecule)
    dyn.run(fmax=0.01)
    res.append((molecule.get_distance(0, 5), molecule.get_potential_energy(), molecule.copy()))





In [ ]:
import matplotlib.pyplot as plt
distances = [r[0] for r in res]
energies = [r[1] - res[0][1] for r in res]
plt.plot(distances, energies)
plt.xlabel('C-Br Distance (Å)')
plt.ylabel('Potential Energy (eV)')
plt.title('SN2 Reaction Energy Profile')
# plt.show()

for i,r in enumerate(res):
    plt.text(distances[i]-0.01, energies[i]+0.01, f'{i}', fontsize=8)

In [ ]:
def get_ts_index(energies):
    # If i-1 < i < i+1, then i is a local maximum
    for i in range(1, len(energies)-1):
        if energies[i-1] < energies[i] and energies[i] > energies[i+1]:
            return i
    return None

ts_guess = res[get_ts_index(energies)]
ts_guess_structure = ts_guess[2]


## 3. Refine an interior maximum with Sella
The highest interior local maximum is used as a starting guess. Remove the scan constraints before
saddle optimization. Convergence of this optimizer is necessary, but does not establish that the
saddle connects the intended reactant and product.


In [ ]:
from sella import Sella
ts_guess_structure.set_constraint()
ts_guess_structure.calc = calculator
opt = Sella(ts_guess_structure, internal=True)
opt.run(1e-4, 1000)

## 4. Validate the saddle using vibrations
For an unconstrained nonlinear molecule, expect one chemically meaningful imaginary frequency at
a first-order saddle, plus near-zero translation/rotation modes. Small imaginary frequencies can be
numerical artifacts
Inspect the displacement: does it form C-Br while breaking C-Cl?

This finite-difference calculation makes multiple force evaluations. Its cache lives in ./vib


In [ ]:
# Verify the transition state by checking vibrational frequencies
from ase.vibrations import Vibrations
vib = Vibrations(ts_guess_structure)
vib.run()
vib.summary()

In [ ]:
vib.write_mode(0)
mode0 = read('vib.0.traj', index=':')
view(mode0)


## 5. Optional: follow both IRC branches
Enable only after inspecting the saddle and unstable mode. Each direction starts from a fresh copy
of the saddle and writes to its own trajectory; we do not infer a branch boundary from an energy jump.
Forward and reverse labels do not inherently identify reactants and products. Inspect both endpoints.
This extension can take substantially longer than the scan.


In [ ]:
# Optionally, do IRC on the transition state
# More information on IRC can be found at:
# https://github.com/zadorlab/sella/wiki/IRC
from sella import IRC
opt = IRC(ts_guess_structure, trajectory='irc.traj', dx=0.1, eta=1e-4, gamma=0.4)
opt.run(fmax=0.1, steps=1000, direction='forward')
opt.run(fmax=0.1, steps=1000, direction='reverse')

## Try, explain, and report
1. Does the unstable mode describe the expected bond exchange?
2. If you ran IRC, do the endpoints correspond to the intended reactant and product sides?
3. What additional evidence would you want before trusting the model for this reaction?

**Checkpoint:** record the charge, model, convergence flag, imaginary frequencies, and endpoint evidence.
A converged saddle on an MLIP surface is not by itself validation against electronic-structure reference data.
See [Sella](https://github.com/zadorlab/sella) and [ASE vibrations](https://docs.ase-lib.org/ase/vibrations/modes.html).
